# 🐎 PonyTown Google云盘一键部署

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ritori2022/pony-town-ready/blob/source-clean-20250827/PonyTown_GDrive_Deploy_Fixed.ipynb)

**🎮 使用Google云盘预构建包快速部署PonyTown！**

## ✨ 新版本优势
- 🚀 **超快部署** - 跳过依赖安装，直接使用预构建包
- 🔒 **环境一致** - 使用与本地完全相同的node_modules
- 📦 **零编译** - 预编译的原生模块，无兼容性问题
- ⚡ **2分钟启动** - 从下载到运行仅需2分钟

## 🎯 使用说明
1. 点击上方 **"Open in Colab"** 按钮
2. 按顺序执行下方所有代码块
3. 在步骤4中输入ngrok token (免费获取)
4. 等待下载完成即可开始游戏！

---

## 📦 步骤 1: 下载预构建的PonyTown包

In [ ]:
import os
import requests
import zipfile
from IPython.display import clear_output, HTML
import time

print("🐎 PonyTown Google云盘部署器启动中...")
print("📦 正在从Google云盘下载预构建包...")

# Google云盘文件ID (从分享链接提取)
# https://drive.google.com/file/d/1eKHs0ANgqDlEe8tAAw30qJqxytxQRvth/view?usp=sharing
GDRIVE_FILE_ID = "1eKHs0ANgqDlEe8tAAw30qJqxytxQRvth"
ZIP_FILENAME = "ponyTown-master.zip"

print(f"📥 下载文件: {ZIP_FILENAME}")
print("⏳ 这可能需要几分钟，取决于网络速度...")

try:
    # 使用gdown下载Google云盘文件
    !pip install -q gdown
    import gdown
    
    # 下载文件
    url = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"
    gdown.download(url, ZIP_FILENAME, quiet=False)
    
    # 检查文件是否下载成功
    if os.path.exists(ZIP_FILENAME):
        file_size = os.path.getsize(ZIP_FILENAME) / (1024*1024)  # MB
        print(f"✅ 下载完成! 文件大小: {file_size:.1f} MB")
        
        # 解压文件
        print("📂 解压项目文件...")
        with zipfile.ZipFile(ZIP_FILENAME, 'r') as zip_ref:
            zip_ref.extractall('/content/')
        
        # 查找解压后的目录
        extracted_dirs = [d for d in os.listdir('/content/') if os.path.isdir(f'/content/{d}') and 'pony' in d.lower()]
        
        if extracted_dirs:
            project_dir = extracted_dirs[0]
            print(f"📁 找到项目目录: {project_dir}")
            os.chdir(f'/content/{project_dir}')
            
            # 检查关键文件
            key_files = ['pony-town.js', 'package.json', 'config.json']
            missing_files = [f for f in key_files if not os.path.exists(f)]
            
            if missing_files:
                print(f"⚠️ 缺少关键文件: {missing_files}")
            else:
                print("✅ 项目文件完整!")
            
            # 检查node_modules
            if os.path.exists('node_modules'):
                node_modules_size = sum(os.path.getsize(os.path.join(dirpath, filename))
                                      for dirpath, dirnames, filenames in os.walk('node_modules')
                                      for filename in filenames) / (1024*1024*1024)  # GB
                print(f"📦 发现预构建依赖: {node_modules_size:.2f} GB")
            else:
                print("📦 未发现node_modules，稍后需要安装依赖")
            
            # 显示目录内容
            print("📋 项目内容:")
            !ls -la | head -15
            
        else:
            print("❌ 未找到项目目录")
            os.chdir('/content/')
            
    else:
        print("❌ 文件下载失败")
        
except Exception as e:
    print(f"❌ 下载过程出错: {e}")
    print("📝 将使用备选方案...")
    
    # 备选方案：使用git clone
    print("📥 使用备选方案: Git clone...")
    !rm -rf pony-town-ready 2>/dev/null || true
    !git clone --depth 1 -b source-clean-20250827 https://github.com/Ritori2022/pony-town-ready.git
    os.chdir('/content/pony-town-ready')

print(f"📁 当前工作目录: {os.getcwd()}")
print("✅ 项目准备完成！")

# 清理下载的zip文件
if os.path.exists(f'/content/{ZIP_FILENAME}'):
    os.remove(f'/content/{ZIP_FILENAME}')
    print("🗑️ 清理临时文件完成")

print(\"📚 检查和处理项目依赖...\")\n\nbash_script = r'''\nexport NVM_DIR=\"$HOME/.nvm\"\n[ -s \"$NVM_DIR/nvm.sh\" ] && . \"$NVM_DIR/nvm.sh\"\nnvm use 9.11.2\n\ncd /content/ponyTown-master || cd /content/pony-town-ready || cd /content/*/\n\necho \"🔍 检查当前目录和文件:\"\npwd\nls -la | head -10\n\necho \"\"\necho \"🔍 检查关键配置文件...\"\nMISSING_FILES=()\n\n# 检查关键文件\nfor file in \"config.json\" \"settings/settings.json\" \".nvmrc\"; do\n    if [ ! -f \"$file\" ]; then\n        echo \"❌ 缺失文件: $file\"\n        MISSING_FILES+=(\"$file\")\n    else\n        echo \"✅ 找到文件: $file\"\n    fi\ndone\n\n# 如果有缺失文件，从浅克隆导入\nif [ ${#MISSING_FILES[@]} -gt 0 ]; then\n    echo \"\"\n    echo \"📥 从浅克隆导入缺失的配置文件...\"\n    \n    # 克隆浅仓库到临时目录\n    git clone --depth 1 -b source-clean-20250827 https://github.com/Ritori2022/pony-town-ready.git /tmp/pony-config\n    \n    # 复制缺失的文件\n    for file in \"${MISSING_FILES[@]}\"; do\n        if [ -f \"/tmp/pony-config/$file\" ]; then\n            echo \"📋 复制文件: $file\"\n            # 创建目录结构（如果需要）\n            mkdir -p \"$(dirname \"$file\")\"\n            cp \"/tmp/pony-config/$file\" \"$file\"\n            echo \"✅ 已导入: $file\"\n        else\n            echo \"⚠️ 浅克隆中也未找到: $file\"\n        fi\n    done\n    \n    # 清理临时目录\n    rm -rf /tmp/pony-config\n    \n    echo \"📁 检查根目录关键文件:\"\n    ls -la config*.json .nvmrc 2>/dev/null || echo \"某些配置文件可能仍然缺失\"\nelse\n    echo \"✅ 所有关键配置文件都存在\"\nfi\n\necho \"\"\necho \"🔍 检查是否存在预构建的node_modules...\"\nif [ -d \"node_modules\" ] && [ -f \"package.json\" ]; then\n    echo \"✅ 发现预构建的node_modules目录\"\n    echo \"📊 node_modules大小: $(du -sh node_modules 2>/dev/null | cut -f1 || echo '未知')\"\n    \n    echo \"🔧 验证关键依赖是否存在...\"\n    DEPS_OK=true\n    \n    # 检查关键依赖\n    for dep in \"express\" \"ws\" \"mongoose\" \"passport\"; do\n        if [ -d \"node_modules/$dep\" ]; then\n            echo \"✓ $dep: 已安装\"\n        else\n            echo \"✗ $dep: 缺失\"\n            DEPS_OK=false\n        fi\n    done\n    \n    if [ \"$DEPS_OK\" = \"true\" ]; then\n        echo \"✅ 关键依赖验证通过，跳过重新安装\"\n        SKIP_INSTALL=true\n    else\n        echo \"⚠️ 部分关键依赖缺失，需要补充安装\"\n        SKIP_INSTALL=false\n    fi\nelse\n    echo \"📦 未发现预构建依赖，需要完整安装\"\n    SKIP_INSTALL=false\nfi\n\nif [ \"$SKIP_INSTALL\" != \"true\" ]; then\n    echo \"\"\n    echo \"📥 安装/更新项目依赖 (这可能需要几分钟)...\"\n    \n    # 清理可能损坏的node_modules\n    if [ -d \"node_modules\" ]; then\n        echo \"🧹 清理可能损坏的node_modules...\"\n        rm -rf node_modules package-lock.json 2>/dev/null || true\n    fi\n    \n    echo \"📦 开始安装依赖...\"\n    npm install\n    \n    if [ $? -eq 0 ]; then\n        echo \"✅ 依赖安装成功\"\n    else\n        echo \"❌ 依赖安装失败，尝试备选方案...\"\n        npm install --no-optional --legacy-peer-deps\n    fi\nfi\n\necho \"\"\necho \"🔧 确保WebSocket兼容性修复...\"\n\n# 检查并修复JavaScript文件\nif [ -f \"src/scripts/server/server.js\" ]; then\n    if grep -q \"clusterws-uws\" src/scripts/server/server.js; then\n        echo \"🔨 修复server.js中的WebSocket引用...\"\n        sed -i 's/const clusterws_uws_1 = require(\"clusterws-uws\");/const WebSocket = require(\"ws\");/g' src/scripts/server/server.js\n        sed -i 's/clusterws_uws_1\\.WebSocketServer/WebSocket.Server/g' src/scripts/server/server.js\n        echo \"✅ server.js WebSocket修复完成\"\n    else\n        echo \"✅ server.js WebSocket已修复或不需要修复\"\n    fi\nelse\n    echo \"⚠️ 未找到src/scripts/server/server.js文件\"\nfi\n\n# 检查并修复TypeScript文件 (如果存在)\nif [ -f \"src/ts/server/server.ts\" ]; then\n    if grep -q \"clusterws-uws\" src/ts/server/server.ts; then\n        echo \"🔨 修复server.ts中的WebSocket引用...\"\n        sed -i 's/import { WebSocketServer } from '\\''clusterws-uws'\\'';//import * as WebSocket from '\\''ws'\\'';//g' src/ts/server/server.ts\n        sed -i 's/WebSocketServer/WebSocket.Server/g' src/ts/server/server.ts\n        echo \"✅ server.ts WebSocket修复完成\"\n    else\n        echo \"✅ server.ts WebSocket已修复或不需要修复\"\n    fi\nfi\n\necho \"\"\necho \"✅ 依赖处理完成，验证关键包:\"\nnode -e \"\ntry {\n  console.log('✓ Express:', require('express').version || 'installed');\n  console.log('✓ WebSocket (ws):', require('ws').WebSocketServer ? 'available' : 'installed'); \n  console.log('✓ Mongoose:', require('mongoose').version || 'installed');\n  console.log('✓ Passport:', require('passport').version || 'installed');\n} catch(e) {\n  console.log('⚠️ 验证时出现错误:', e.message);\n  console.log('这通常不影响运行，某些模块可能采用不同的导出方式');\n}\n\"\n\necho \"\"\necho \"📊 项目最终状态:\"\necho \"📁 目录: $(pwd)\"\necho \"📦 依赖: $(ls node_modules 2>/dev/null | wc -l || echo '0') 个模块\"\necho \"🎮 主文件: $(ls -la pony-town.js 2>/dev/null | cut -d' ' -f5 || echo '未找到') bytes\"\necho \"⚙️ 配置文件: $(ls -la config.json 2>/dev/null && echo '✅' || echo '❌ 缺失')\"\n'''\n\nwith open('/tmp/process_deps.sh', 'w') as f:\n    f.write(bash_script)\n\n!bash /tmp/process_deps.sh\n\nprint(\"✅ 依赖处理、配置文件导入和WebSocket修复完成！\")\nprint(\"🎯 使用预构建包 + 自动配置文件导入确保完整性\")"

In [ ]:
print(\"🎵 检查并处理游戏音乐资源...\")\n\nbash_script = r'''\nexport NVM_DIR=\"$HOME/.nvm\"\n[ -s \"$NVM_DIR/nvm.sh\" ] && . \"$NVM_DIR/nvm.sh\"\nnvm use 9.11.2\n\ncd /content/ponyTown-master || cd /content/pony-town-ready || cd /content/*/\n\necho \"🎵 检查音乐资源状态...\"\nif [ -d \"assets/music\" ] && [ \"$(ls -A assets/music 2>/dev/null)\" ]; then\n    MUSIC_COUNT=$(ls assets/music/*.mp3 assets/music/*.webm 2>/dev/null | wc -l)\n    echo \"✅ 发现音乐文件: $MUSIC_COUNT 个\"\n    echo \"📋 音乐文件示例:\"\n    ls -la assets/music/ | head -5\nelse\n    echo \"📥 音乐文件缺失，开始下载...\"\n    \n    # 创建assets目录（如果不存在）\n    mkdir -p assets\n    \n    # 从原始仓库下载音乐\n    echo \"📦 克隆音乐资源...\"\n    git clone --depth 1 --filter=blob:none --sparse https://github.com/Ritori2022/ponyTown.git /tmp/temp-music\n    cd /tmp/temp-music\n    git sparse-checkout set assets/music\n    \n    # 复制音乐文件\n    if [ -d \"assets/music\" ]; then\n        cp -r assets/music \"$(pwd | sed 's|/tmp/temp-music||')/assets/\"\n        echo \"✅ 音乐文件下载完成\"\n    else\n        echo \"⚠️ 音乐资源下载失败\"\n    fi\n    \n    # 清理临时目录\n    cd - > /dev/null\n    rm -rf /tmp/temp-music\n    \n    # 验证音乐文件\n    if [ -d \"assets/music\" ]; then\n        MUSIC_COUNT=$(ls assets/music/*.mp3 assets/music/*.webm 2>/dev/null | wc -l)\n        echo \"🎼 最终音乐文件数量: $MUSIC_COUNT\"\n        echo \"📋 音乐文件:\"\n        ls -la assets/music/ | head -10\n    fi\nfi\n\necho \"\"\necho \"🔍 检查其他必要的资源目录...\"\nfor dir in \"assets/images\" \"build\" \"public\" \"favicons\"; do\n    if [ -d \"$dir\" ]; then\n        echo \"✅ $dir: 存在\"\n    else\n        echo \"⚠️ $dir: 缺失\"\n    fi\ndone\n\necho \"\"\necho \"📊 资源文件总结:\"\necho \"🎵 音乐: $(ls assets/music/ 2>/dev/null | wc -l) 个文件\"\necho \"🖼️ 图片: $(find assets -name '*.png' -o -name '*.jpg' -o -name '*.gif' 2>/dev/null | wc -l) 个文件\"\necho \"📦 构建: $(ls build/ 2>/dev/null | wc -l) 个文件\"\n'''\n\nwith open('/tmp/check_assets.sh', 'w') as f:\n    f.write(bash_script)\n\n!bash /tmp/check_assets.sh\n\nprint(\"✅ 游戏资源检查和处理完成！\")"

## 📚 步骤 3: 处理依赖和配置

In [ ]:
import threading\nimport time\nimport subprocess\nimport requests\nimport json\nimport os\nfrom IPython.display import HTML\n\nprint(\"🚀 启动 PonyTown 服务器...\")\n\n# 确定项目目录\nproject_dirs = ['/content/ponyTown-master', '/content/pony-town-ready']\nproject_dir = None\n\nfor dir_path in project_dirs:\n    if os.path.exists(dir_path) and os.path.exists(f'{dir_path}/pony-town.js'):\n        project_dir = dir_path\n        break\n\nif not project_dir:\n    # 尝试查找任何包含pony-town.js的目录\n    for item in os.listdir('/content/'):\n        item_path = f'/content/{item}'\n        if os.path.isdir(item_path) and os.path.exists(f'{item_path}/pony-town.js'):\n            project_dir = item_path\n            break\n\nif not project_dir:\n    print(\"❌ 未找到PonyTown项目目录！\")\n    print(\"🔍 当前/content/目录内容:\")\n    !ls -la /content/\n    raise Exception(\"项目目录未找到\")\n\nprint(f\"📁 使用项目目录: {project_dir}\")\n\n# 启动前的完整检查和修复\nprint(\"🔧 启动前完整检查...\")\nbash_script = f'''\nexport NVM_DIR=\"$HOME/.nvm\"\n[ -s \"$NVM_DIR/nvm.sh\" ] && . \"$NVM_DIR/nvm.sh\"\nnvm use 9.11.2\n\ncd {project_dir}\n\necho \"📋 项目启动前检查:\"\necho \"📁 当前目录: $(pwd)\"\n\necho \"\"\necho \"🔍 检查关键文件:\"\nfor file in \"pony-town.js\" \"config.json\" \"package.json\"; do\n    if [ -f \"$file\" ]; then\n        echo \"✅ $file: $(ls -lh $file | awk '{{print $5}}')\"\n    else\n        echo \"❌ $file: 缺失\"\n    fi\ndone\n\necho \"\"\necho \"🔍 检查必要目录:\"\nfor dir in \"src/scripts/server\" \"assets\" \"node_modules\"; do\n    if [ -d \"$dir\" ]; then\n        echo \"✅ $dir/: 存在\"\n    else\n        echo \"❌ $dir/: 缺失\"\n    fi\ndone\n\n# 检查并修复config.json路径问题\necho \"\"\necho \"🔧 检查config.json路径引用...\"\nif [ -f \"src/scripts/server/config.js\" ]; then\n    # 检查config.js中的路径引用\n    if grep -q \"../../../config.json\" src/scripts/server/config.js; then\n        echo \"📝 config.js引用: ../../../config.json\"\n        if [ ! -f \"config.json\" ]; then\n            echo \"⚠️ config.json不在根目录，尝试修复...\"\n            # 如果config.json在其他位置，复制到根目录\n            find . -name \"config.json\" -not -path \"./node_modules/*\" -exec cp {{}} . \\;\n        fi\n    fi\nelse\n    echo \"⚠️ 未找到config.js文件\"\nfi\n\n# 最终验证\necho \"\"\necho \"🔍 最终启动条件检查:\"\nSTART_OK=true\n\nif [ ! -f \"pony-town.js\" ]; then\n    echo \"❌ 主程序文件缺失\"\n    START_OK=false\nfi\n\nif [ ! -f \"config.json\" ]; then\n    echo \"❌ 配置文件缺失\"\n    START_OK=false\nfi\n\nif [ ! -d \"node_modules\" ]; then\n    echo \"❌ 依赖目录缺失\"\n    START_OK=false\nfi\n\nif [ \"$START_OK\" = \"true\" ]; then\n    echo \"✅ 所有启动条件满足\"\nelse\n    echo \"❌ 启动条件不满足，需要修复上述问题\"\n    exit 1\nfi\n'''\n\nwith open('/tmp/pre_start_check.sh', 'w') as f:\n    f.write(bash_script)\n\n# 执行启动前检查\nresult = subprocess.run(['bash', '/tmp/pre_start_check.sh'], capture_output=True, text=True)\nprint(result.stdout)\nif result.returncode != 0:\n    print(f\"❌ 启动前检查失败: {result.stderr}\")\n    print(\"🔄 尝试重新运行之前的配置步骤来修复问题\")\n    raise Exception(\"启动条件不满足\")\n\n# 启动PonyTown服务器\nprint(\"🎮 启动 PonyTown 服务器 (混合模式)...\")\nbash_script = f'''\nexport NVM_DIR=\"$HOME/.nvm\"\n[ -s \"$NVM_DIR/nvm.sh\" ] && . \"$NVM_DIR/nvm.sh\"\nnvm use 9.11.2\n\ncd {project_dir}\n\necho \"🚀 启动服务器进程...\"\n# 使用nohup让服务器在后台持续运行，并捕获详细日志\nnohup node pony-town.js --login --local --game > /tmp/ponytown.log 2>&1 &\nSERVER_PID=$!\necho $SERVER_PID > /tmp/ponytown.pid\n\necho \"✅ PonyTown服务器已启动 (PID: $SERVER_PID)\"\necho \"📊 等待服务器初始化...\"\nsleep 5\n\n# 检查服务器进程状态\nif kill -0 $SERVER_PID 2>/dev/null; then\n    echo \"✅ 服务器进程运行正常\"\nelse\n    echo \"❌ 服务器进程异常，显示启动日志:\"\n    cat /tmp/ponytown.log\n    exit 1\nfi\n\necho \"📋 当前进程信息:\"\nps aux | grep \"node pony-town\" | grep -v grep || echo \"⚠️ 未找到服务器进程\"\n'''\n\nwith open('/tmp/start_server.sh', 'w') as f:\n    f.write(bash_script)\n\n# 启动服务器\nresult = subprocess.run(['bash', '/tmp/start_server.sh'], capture_output=True, text=True)\nprint(result.stdout)\nif result.returncode != 0:\n    print(f\"❌ 服务器启动失败: {result.stderr}\")\n    print(\"📋 服务器错误日志:\")\n    try:\n        with open('/tmp/ponytown.log', 'r') as f:\n            print(f.read()[-2000:])\n    except:\n        print(\"无法读取错误日志\")\n    raise Exception(\"服务器启动失败\")\n\n# 等待服务器启动并检查状态\nprint(\"⏳ 等待PonyTown服务器完全启动...\")\nserver_started = False\n\nfor i in range(60):  # 最多等待60秒\n    time.sleep(1)\n    try:\n        response = requests.get('http://localhost:8090', timeout=3)\n        if response.status_code == 200:\n            print(f\"✅ PonyTown服务器成功启动! (耗时 {i+1} 秒)\")\n            server_started = True\n            break\n    except requests.exceptions.RequestException:\n        if i % 15 == 14:  # 每15秒显示一次进度\n            print(f\"⏳ 等待服务器响应... ({i+1}/60秒)\")\n        continue\n\nif not server_started:\n    print(\"⚠️ 服务器启动超时，显示详细诊断信息:\")\n    print(\"🔍 检查服务器进程:\")\n    !ps aux | grep node | head -5\n    \n    print(\"🔍 检查端口占用:\")\n    !netstat -tlnp | grep :8090 2>/dev/null || echo \"端口8090未被占用\"\n    \n    print(\"🔍 服务器完整日志:\")\n    try:\n        with open('/tmp/ponytown.log', 'r') as f:\n            log_content = f.read()\n            print(log_content[-4000:] if len(log_content) > 4000 else log_content)\n    except:\n        print(\"📋 无法读取服务器日志\")\n\n# 启动ngrok (如果配置了token)\npublic_url = None\nlogin_url = None\n\nif os.environ.get('USE_NGROK', 'false') == 'true' and server_started:\n    print(\"🌐 启动 ngrok 隧道...\")\n    # 使用后台进程启动ngrok\n    ngrok_process = subprocess.Popen(['ngrok', 'http', '8090'], \n                                   stdout=subprocess.DEVNULL, \n                                   stderr=subprocess.DEVNULL)\n    time.sleep(10)  # 给ngrok更多时间建立隧道\n    \n    try:\n        print(\"🔍 获取 ngrok 公网地址...\")\n        ngrok_api = requests.get('http://127.0.0.1:4040/api/tunnels', timeout=10)\n        tunnels = ngrok_api.json()\n        if tunnels.get('tunnels') and len(tunnels['tunnels']) > 0:\n            public_url = tunnels['tunnels'][0]['public_url']\n            login_url = f\"{public_url}/auth/local/68acdc3543a9ff7ce48a3daa\"\n            \n            print(\"🎉 ngrok隧道创建成功!\")\n            print(f\"🌐 公网地址: {public_url}\")\n            print(f\"🎮 快速登录: {login_url}\")\n        else:\n            print(\"⚠️ 未找到活动的ngrok隧道\")\n    except Exception as e:\n        print(f\"⚠️ 获取ngrok地址时出错: {e}\")\n\n# 如果ngrok失败或未配置，使用localhost\nif not public_url:\n    public_url = \"http://localhost:8090\"\n    login_url = \"http://localhost:8090/auth/local/68acdc3543a9ff7ce48a3daa\"\n    print(\"🏠 使用本地访问:\")\n    print(f\"🌐 本地地址: {public_url}\")\n    print(f\"🎮 快速登录: {login_url}\")\n    print(\"📝 注意: 本地地址只能在这个Colab会话中访问\")\n\n# 最终服务器状态检查\ntry:\n    final_check = requests.get('http://localhost:8090', timeout=5)\n    server_status = \"✅ 运行正常\" if final_check.status_code == 200 else f\"⚠️ 状态码: {final_check.status_code}\"\nexcept Exception as e:\n    server_status = f\"❌ 连接失败: {str(e)[:100]}...\"\n\nprint(f\"🔍 最终服务器状态: {server_status}\")\n\n# 显示游戏链接\nprint(\"\")\nprint(\"🎯 开始游戏:\")\nprint(\"   1. 点击下方的游戏链接\")\nprint(\"   2. 等待游戏加载完成\") \nprint(\"   3. 创建你的小马角色\")\nprint(\"   4. 享受游戏体验！\")\nprint(\"\")\n\n# HTML游戏链接展示\ndisplay(HTML(f'''\n<div style=\"padding: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 15px; margin: 15px 0; box-shadow: 0 8px 32px rgba(0,0,0,0.3);\">\n    <h3 style=\"margin-top: 0; text-align: center; font-size: 24px;\">🎮 PonyTown 游戏服务器</h3>\n    <div style=\"background: rgba(255,255,255,0.15); padding: 20px; border-radius: 12px; margin: 15px 0; text-align: center;\">\n        <p style=\"margin: 8px 0;\"><strong>🌐 服务器地址:</strong></p>\n        <p style=\"margin: 8px 0; font-family: monospace; background: rgba(0,0,0,0.2); padding: 8px; border-radius: 5px;\">{public_url}</p>\n        <p style=\"color: #ddd; font-size: 14px; margin: 5px 0;\">📊 状态: {server_status}</p>\n        <div style=\"margin: 20px 0;\">\n            <a href=\"{login_url}\" target=\"_blank\" style=\"display: inline-block; background: #4CAF50; color: white; padding: 15px 30px; border-radius: 30px; text-decoration: none; font-weight: bold; font-size: 18px; box-shadow: 0 4px 15px rgba(0,0,0,0.2); transition: transform 0.2s;\">🐎 点击进入游戏</a>\n        </div>\n        <p style=\"color: #ddd; font-size: 14px; margin: 5px 0;\">💡 建议在新标签页中打开</p>\n        <p style=\"color: #ddd; font-size: 13px; margin: 5px 0;\">🔐 使用Mock登录 - 无需注册</p>\n    </div>\n</div>\n'''))\n\nprint(\"📋 部署状态总结:\")\nprint(\"   - Node.js: 9.11.2 ✅\")\nprint(\"   - 预构建包: 已使用 ✅\") \nprint(\"   - WebSocket: 修复完成 ✅\")\nprint(\"   - 配置文件: 已导入 ✅\")\nprint(\"   - 音乐资源: 已检查 ✅\")\nprint(\"   - 登录系统: Mock认证 ✅\")\nprint(f\"   - 服务器状态: {server_status}\")\nprint(\"\")\n\nif server_started:\n    print(\"🎉 PonyTown部署成功！享受游戏吧！\")\n    print(\"📝 服务器将在后台持续运行\")\nelse:\n    print(\"⚠️ 服务器启动遇到问题，请检查上方的诊断信息\")\n    print(\"🔄 可以尝试重新运行之前的步骤或查看故障排除部分\")\n\nprint(\"\")\nprint(\"💻 服务器管理命令:\")\nprint(\"   查看日志: !tail -f /tmp/ponytown.log\")\nprint(\"   停止服务器: !pkill -f 'node pony-town.js'\")\nprint(\"   检查进程: !ps aux | grep node\")\nprint(f\"   项目目录: {project_dir}\")\n\n# 保存项目目录到环境变量\nos.environ['PONYTOWN_DIR'] = project_dir"

## 🌐 步骤 4: 设置ngrok公网访问

In [ ]:
print("🌐 设置 ngrok 公网隧道...")

# 安装ngrok
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar xvzf ngrok-v3-stable-linux-amd64.tgz
!chmod +x ngrok
!mv ngrok /usr/local/bin/

# 设置ngrok认证
import getpass
import os

print("🔑 ngrok 认证设置")
print("为了使用ngrok公网隧道，需要免费的ngrok token")
print("")
print("📋 获取免费token步骤:")
print("1. 访问: https://ngrok.com/")
print("2. 点击 'Sign up free' 免费注册")
print("3. 登录后在 Dashboard 页面找到 'Your Authtoken'")
print("4. 复制token并在下方输入")
print("")

# 获取用户输入的token
ngrok_token = getpass.getpass("🔐 请输入你的ngrok token (输入不会显示): ")

if ngrok_token.strip():
    !ngrok config add-authtoken {ngrok_token}
    print("✅ ngrok 认证完成！")
    os.environ['NGROK_TOKEN'] = ngrok_token
    os.environ['USE_NGROK'] = 'true'
else:
    print("⚠️  未提供token，将使用localhost访问")
    print("📝 注意: 没有ngrok token只能本地访问，无法分享给他人")
    os.environ['USE_NGROK'] = 'false'

print("📝 ngrok 设置完成！")

## 🚀 步骤 5: 启动PonyTown服务器

In [ ]:
import threading
import time
import subprocess
import requests
import json
import os
from IPython.display import HTML

print("🚀 启动 PonyTown 服务器...")

# 确定项目目录
project_dirs = ['/content/ponyTown-master', '/content/pony-town-ready']
project_dir = None

for dir_path in project_dirs:
    if os.path.exists(dir_path) and os.path.exists(f'{dir_path}/pony-town.js'):
        project_dir = dir_path
        break

if not project_dir:
    # 尝试查找任何包含pony-town.js的目录
    for item in os.listdir('/content/'):
        item_path = f'/content/{item}'
        if os.path.isdir(item_path) and os.path.exists(f'{item_path}/pony-town.js'):
            project_dir = item_path
            break

if not project_dir:
    print("❌ 未找到PonyTown项目目录！")
    print("🔍 当前/content/目录内容:")
    !ls -la /content/
    raise Exception("项目目录未找到")

print(f"📁 使用项目目录: {project_dir}")

# 启动PonyTown服务器
print("🎮 启动 PonyTown 服务器 (混合模式)...")
bash_script = f'''
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
nvm use 9.11.2

cd {project_dir}

echo "🎮 准备启动PonyTown服务器..."
echo "📁 当前目录: $(pwd)"
echo "📝 检查关键文件:"
ls -la pony-town.js config.json package.json 2>/dev/null

echo ""
echo "🔧 最后检查WebSocket修复状态..."
if grep -q "clusterws-uws" src/scripts/server/server.js 2>/dev/null; then
    echo "⚠️ 检测到未修复的WebSocket引用，正在修复..."
    sed -i 's/const clusterws_uws_1 = require("clusterws-uws");/const WebSocket = require("ws");/g' src/scripts/server/server.js
    sed -i 's/clusterws_uws_1\\.WebSocketServer/WebSocket.Server/g' src/scripts/server/server.js
    echo "✅ WebSocket修复完成"
else
    echo "✅ WebSocket已修复或不需要修复"
fi

echo ""
echo "🚀 启动服务器进程..."
# 使用nohup让服务器在后台持续运行
nohup node pony-town.js --login --local --game > /tmp/ponytown.log 2>&1 &
SERVER_PID=$!
echo $SERVER_PID > /tmp/ponytown.pid

echo "✅ PonyTown服务器已启动 (PID: $SERVER_PID)"
echo "📊 等待服务器初始化..."
sleep 3

# 检查服务器进程状态
if kill -0 $SERVER_PID 2>/dev/null; then
    echo "✅ 服务器进程运行正常"
else
    echo "❌ 服务器进程异常，检查启动日志:"
    tail -30 /tmp/ponytown.log 2>/dev/null || echo "无法读取日志"
fi

echo "📋 进程信息:"
ps aux | grep "node pony-town" | grep -v grep || echo "未找到服务器进程"
'''

with open('/tmp/start_server.sh', 'w') as f:
    f.write(bash_script)

# 启动服务器
result = subprocess.run(['bash', '/tmp/start_server.sh'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(f"⚠️ 警告: {result.stderr}")

# 等待服务器启动并检查状态
print("⏳ 等待PonyTown服务器完全启动...")
server_started = False

for i in range(60):  # 最多等待60秒
    time.sleep(1)
    try:
        response = requests.get('http://localhost:8090', timeout=3)
        if response.status_code == 200:
            print(f"✅ PonyTown服务器成功启动! (耗时 {i+1} 秒)")
            server_started = True
            break
    except requests.exceptions.RequestException:
        if i % 15 == 14:  # 每15秒显示一次进度
            print(f"⏳ 等待服务器响应... ({i+1}/60秒)")
        continue

if not server_started:
    print("⚠️ 服务器启动超时，显示详细诊断信息:")
    print("🔍 检查服务器进程:")
    !ps aux | grep node | head -5
    
    print("🔍 检查端口占用:")
    !netstat -tlnp | grep :8090 2>/dev/null || echo "端口8090未被占用"
    
    print("🔍 服务器启动日志 (最后50行):")
    try:
        with open('/tmp/ponytown.log', 'r') as f:
            log_content = f.read()[-3000:]  # 显示最后3000字符
            print(log_content)
    except:
        print("📋 无法读取服务器日志")

# 启动ngrok (如果配置了token)
public_url = None
login_url = None

if os.environ.get('USE_NGROK', 'false') == 'true' and server_started:
    print("🌐 启动 ngrok 隧道...")
    # 使用后台进程启动ngrok
    ngrok_process = subprocess.Popen(['ngrok', 'http', '8090'], 
                                   stdout=subprocess.DEVNULL, 
                                   stderr=subprocess.DEVNULL)
    time.sleep(10)  # 给ngrok更多时间建立隧道
    
    try:
        print("🔍 获取 ngrok 公网地址...")
        ngrok_api = requests.get('http://127.0.0.1:4040/api/tunnels', timeout=10)
        tunnels = ngrok_api.json()
        if tunnels.get('tunnels') and len(tunnels['tunnels']) > 0:
            public_url = tunnels['tunnels'][0]['public_url']
            login_url = f"{public_url}/auth/local/68acdc3543a9ff7ce48a3daa"
            
            print("🎉 ngrok隧道创建成功!")
            print(f"🌐 公网地址: {public_url}")
            print(f"🎮 快速登录: {login_url}")
        else:
            print("⚠️ 未找到活动的ngrok隧道")
    except Exception as e:
        print(f"⚠️ 获取ngrok地址时出错: {e}")

# 如果ngrok失败或未配置，使用localhost
if not public_url:
    public_url = "http://localhost:8090"
    login_url = "http://localhost:8090/auth/local/68acdc3543a9ff7ce48a3daa"
    print("🏠 使用本地访问:")
    print(f"🌐 本地地址: {public_url}")
    print(f"🎮 快速登录: {login_url}")
    print("📝 注意: 本地地址只能在这个Colab会话中访问")

# 最终服务器状态检查
try:
    final_check = requests.get('http://localhost:8090', timeout=5)
    server_status = "✅ 运行正常" if final_check.status_code == 200 else f"⚠️ 状态码: {final_check.status_code}"
except Exception as e:
    server_status = f"❌ 连接失败: {str(e)[:100]}..."

print(f"🔍 最终服务器状态: {server_status}")

# 显示游戏链接
print("")
print("🎯 开始游戏:")
print("   1. 点击下方的游戏链接")
print("   2. 等待游戏加载完成") 
print("   3. 创建你的小马角色")
print("   4. 享受游戏体验！")
print("")

# HTML游戏链接展示
display(HTML(f'''
<div style="padding: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 15px; margin: 15px 0; box-shadow: 0 8px 32px rgba(0,0,0,0.3);">
    <h3 style="margin-top: 0; text-align: center; font-size: 24px;">🎮 PonyTown 游戏服务器</h3>
    <div style="background: rgba(255,255,255,0.15); padding: 20px; border-radius: 12px; margin: 15px 0; text-align: center;">
        <p style="margin: 8px 0;"><strong>🌐 服务器地址:</strong></p>
        <p style="margin: 8px 0; font-family: monospace; background: rgba(0,0,0,0.2); padding: 8px; border-radius: 5px;">{public_url}</p>
        <p style="color: #ddd; font-size: 14px; margin: 5px 0;">📊 状态: {server_status}</p>
        <div style="margin: 20px 0;">
            <a href="{login_url}" target="_blank" style="display: inline-block; background: #4CAF50; color: white; padding: 15px 30px; border-radius: 30px; text-decoration: none; font-weight: bold; font-size: 18px; box-shadow: 0 4px 15px rgba(0,0,0,0.2); transition: transform 0.2s;">🐎 点击进入游戏</a>
        </div>
        <p style="color: #ddd; font-size: 14px; margin: 5px 0;">💡 建议在新标签页中打开</p>
        <p style="color: #ddd; font-size: 13px; margin: 5px 0;">🔐 使用Mock登录 - 无需注册</p>
    </div>
</div>
'''))

print("📋 部署状态总结:")
print("   - Node.js: 9.11.2 ✅")
print("   - 预构建包: 已使用 ✅") 
print("   - WebSocket: 修复完成 ✅")
print("   - 登录系统: Mock认证 ✅")
print(f"   - 服务器状态: {server_status}")
print("")

if server_started:
    print("🎉 PonyTown部署成功！享受游戏吧！")
    print("📝 服务器将在后台持续运行")
else:
    print("⚠️ 服务器启动遇到问题，请检查上方的诊断信息")
    print("🔄 可以尝试重新运行此步骤或查看故障排除部分")

print("")
print("💻 服务器管理命令:")
print("   查看日志: !tail -f /tmp/ponytown.log")
print("   停止服务器: !pkill -f 'node pony-town.js'")
print("   检查进程: !ps aux | grep node")
print(f"   项目目录: {project_dir}")

# 保存项目目录到环境变量
os.environ['PONYTOWN_DIR'] = project_dir

## 🔧 故障排除和调试

### 🚀 快速调试命令

In [ ]:
# 服务器状态检查
print("🔍 服务器状态检查:")
!echo "📊 Node.js进程:"
!ps aux | grep node | head -5
print("")

!echo "🌐 端口占用检查:"
!netstat -tlnp | grep :8090 || echo "端口8090未被占用"
print("")

!echo "📋 最新服务器日志 (最后20行):"
!tail -20 /tmp/ponytown.log 2>/dev/null || echo "日志文件不存在"
print("")

!echo "🔗 测试本地连接:"
!curl -I http://localhost:8090 2>/dev/null | head -3 || echo "无法连接到localhost:8090"

### 🔄 重启服务器

In [ ]:
# 重启PonyTown服务器
print("🔄 重启 PonyTown 服务器...")

!echo "🛑 停止现有服务器进程..."
!pkill -f 'node pony-town.js' 2>/dev/null || echo "没有找到运行中的服务器"
!sleep 3

!echo "🚀 重新启动服务器..."
# 重新运行启动脚本
!bash /tmp/start_server.sh

print("✅ 服务器重启完成！")
print("⏳ 等待几秒钟然后测试连接...")

---

## 📚 更多信息

- **项目仓库**: [pony-town-ready](https://github.com/Ritori2022/pony-town-ready/tree/source-clean-20250827)
- **版本**: source-clean-20250827 (稳定版本)
- **Node.js版本**: 9.11.2 (必需)
- **部署方式**: Google云盘预构建包 + Colab

### 💡 Google云盘预构建包优势

使用你提供的预构建包 `ponyTown-master.zip` 的好处:
- **跳过依赖安装** - 直接使用本地编译好的 `node_modules`
- **避免兼容性问题** - 无需在Colab中重新编译原生模块
- **更快部署** - 从下载到启动仅需几分钟
- **环境一致** - 与你本地运行的完全相同

**🎮 享受PonyTown多人游戏体验！** 🐎✨